In [ ]:
import pandas as pd
import numpy as np
import random
import re
import itertools
from itertools import product, combinations
import os
import pickle
from collections import defaultdict, Counter
import warnings
import json
import time
import importlib
from datetime import datetime
import copy
from copy import deepcopy
from itertools import cycle
import math
import inspect
from IPython.display import display
from pandas.api.types import is_numeric_dtype
from scipy.stats import fisher_exact, spearmanr, wilcoxon, norm, chi2_contingency
import statsmodels.api as sm

import base_utils; importlib.reload(base_utils)
from base_utils import *

pd.set_option('display.max_columns', 100); pd.set_option('display.max_rows', 145)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"; os.environ['PYTHONHASHSEED'] = '0'

import torch
torch.use_deterministic_algorithms(True)

from xgboost import XGBClassifier

hgnc = get_hgnc26()

In [2]:
def get_data(filename, dp, lower=False, clean=True, **kwargs):
    kind = {'csv':'csv', 'txt':'raw txt', 'tsv':'tsv'}[filename.split('.')[1]] if '.' in filename else 'raw txt'
    df = load_data(kind, filename, dp, clean=clean, comment='#', **kwargs)
    if lower: df.columns = [col.lower() for col in df.columns]
    return df

# https://docs.cancergenomicscloud.org/docs/tcga-metadata

def col_stats(df, valset_max=10, combined_col=False, save=None):
    statdf = pd.DataFrame(index=df.columns.tolist())
    def example(c):
        i = df[c].first_valid_index()
        return np.nan if i is None else df.loc[i, c]
    statdf = statdf.assign(
        n_unique=[df[c].nunique() for c in statdf.index],
        n_nan=[df[c].isnull().sum() for c in statdf.index],
        pct_nan=[round((df[c].isnull().sum()/len(df[c]))*100) for c in statdf.index],
    )
    if combined_col:
        statdf['nan_n_pct'] = [f"{statdf.loc[i, 'n_nan']} ({statdf.loc[i, 'pct_nan']})" for i in statdf.index]
        statdf = statdf.drop(columns=['n_nan', 'pct_nan'])
    statdf['valueset'] = [df[c].unique() if statdf.loc[c, 'n_unique'] <= valset_max else f"[{df[c].unique()[0]}, {df[c].unique()[1]}, ...]" for c in statdf.index]
    if save is not None:
        statdf['valueset'] = [f"[{', '.join(map(str, statdf.loc[i, 'valueset']))}]" if statdf.loc[i, 'n_unique'] <= valset_max else statdf.loc[i, 'valueset'] for i in statdf.index]
        statdf.to_csv(save)
    return statdf

In [ ]:
import lung_data_prep; importlib.reload(lung_data_prep)
from lung_data_prep import TCGA, TRACERx, format_for_modeling, LUNG, GenePrior, Table1, concordance_filter
warnings.simplefilter('error', pd.errors.PerformanceWarning)

ignore = {
    'data_clinical_patient': ['SUBTYPE', 'CANCER_TYPE_ACRONYM', 'OTHER_PATIENT_ID', 'FORM_COMPLETION_DATE', 'INFORMED_CONSENT_VERIFIED', 'IN_PANCANPATHWAYS_FREEZE', 'OS_STATUS', 'OS_MONTHS', 'HISTOLOGICAL_DIAGNOSIS', 'PROSPECTIVE_COLLECTION', 'RETROSPECTIVE_COLLECTION', 'TUMOR_SITE', 'DISEASE_CODE', 'PROJECT_CODE', 'PRIMARY_SITE_OTHER', 'PERFORMANCE_STATUS_TIMING', 'PULMONARY_FUNCTION_TEST_INDICATOR'], # 'INITIAL_PATHOLOGIC_DX_YEAR', 'DAYS_TO_INITIAL_PATHOLOGIC_DIAGNOSIS',
    'data_clinical_sample': ['CANCER_TYPE', 'CANCER_TYPE_DETAILED', 'ONCOTREE_CODE', 'TISSUE_PROSPECTIVE_COLLECTION_INDICATOR', 'TISSUE_RETROSPECTIVE_COLLECTION_INDICATOR', 'TUMOR_TISSUE_SITE', 'TISSUE_SOURCE_SITE_CODE', 'SOMATIC_STATUS', 'OTHER_SAMPLE_ID', 'SPECIMEN_SECOND_LONGEST_DIMENSION', 'IS_FFPE', 'OCT_EMBEDDED', 'PATHOLOGY_REPORT_FILE_NAME', 'PATHOLOGY_REPORT_UUID', 'SHORTEST_DIMENSION', 'VIAL_NUMBER', 'FORM_COMPLETION_DATE', 'INITIAL_PATHOLOGIC_DX_YEAR', 'PROSPECTIVE_COLLECTION', 'RETROSPECTIVE_COLLECTION', 'DAYS_TO_COLLECTION',], # 'DAYS_TO_INITIAL_PATHOLOGIC_DIAGNOSIS', 
    'luad_gdc_phenotypes': ['id', 'disease_type', 'case_id', 'primary_site', 'vital_status.demographic', 'year_of_birth.demographic', 'year_of_death.demographic', 'primary_site.project', 'project_id.project', 'name.project', 'name.program.project', 'tissue_source_site_id.tissue_source_site', 'code.tissue_source_site', 'name.tissue_source_site', 'project.tissue_source_site', 'bcr_id.tissue_source_site', 'submitter_id.annotations', 'classification.annotations', 'entity_id.annotations', 'created_datetime.annotations', 'annotation_id.annotations', 'entity_type.annotations', 'updated_datetime.annotations', 'case_id.annotations', 'state.annotations', 'status.annotations', 'case_submitter_id.annotations', 'site_of_resection_or_biopsy.diagnoses', 'treatment_id.treatments.diagnoses', 'submitter_id.treatments.diagnoses', 'created_datetime.treatments.diagnoses', 'updated_datetime.treatments.diagnoses', 'state.treatments.diagnoses', 'sample_id.samples', 'days_to_collection.samples', 'preservation_method.samples', 'intermediate_dimension.samples', 'pathology_report_uuid.samples', 'shortest_dimension.samples', 'oct_embedded.samples', 'specimen_type.samples', 'is_ffpe.samples', 'tissue_type.samples', 'composition.samples', 'initial_weight.samples', 'longest_dimension.samples'], # 'year_of_diagnosis.diagnoses'
    'xena_phenotypes': ['WGS_as_of_20120731_0_no_1_yes', '_PATIENT', '_cohort', '_primary_site', 'anatomic_neoplasm_subdivision_other', 'bcr_followup_barcode', 'days_to_collection', 'disease_code', 'form_completion_date', 'informed_consent_verified', 'intermediate_dimension', 'is_ffpe', 'egfr_mutation_performed', 'kras_gene_analysis_performed', 'eml4_alk_translocation_performed', 'eml4_alk_translocation_method', 'oct_embedded', 'pathology_report_file_name', 'performance_status_scale_timing', 'progression_determined_by', 'project_code', 'pulmonary_function_test_performed', 'shortest_dimension', 'tissue_prospective_collection_indicator', 'tissue_retrospective_collection_indicator', 'tumor_tissue_site', 'vial_number', 'vital_status', 'Cnncl_mt_n_KRAS_EGFR_ALK_RET_ROS1_BRAF_ERBB2_HRAS_NRAS_AKT1_MAP2', 'Estimated_allele_fraction_of_a_clonal_varnt_prsnt_t_1_cpy_pr_cll', 'Canonical_mut_in_KRAS_EGFR_ALK', 'kras_mutation_found', 'ALK_translocation', 'HRAS', 'NRAS', 'ROS1_translocation'], # 'year_of_initial_pathologic_diagnosis', 'days_to_initial_pathologic_diagnosis', 
    'xena_endpoints': ['_PATIENT', 'OS', 'OS.time'],
    'cdr': ['vital_status', 'OS', 'OS.time'], # 'initial_pathologic_dx_year'
    'followup': ['bcr_patient_uuid', 'form_completion_date', 'informed_consent_verified', 'vital_status', 'egfr_mutation_performed', 'eml4_alk_translocation_performed', 'eml4_alk_translocation_method', 'kras_gene_analysis_performed', 'tissue_prospective_collection_indicator', 'days_to_initial_pathologic_diagnosis', 'tissue_retrospective_collection_indicator', 'performance_status_scale_timing', 'project_code', 'pulmonary_function_test_performed'] # 'year_of_initial_pathologic_diagnosis',  
}
prior = GenePrior(hgnc)

In [14]:
tcga_c = TCGA(ignore, hgnc, cutoff=730, window=730, req_NT=True, window_positives=False, print_excl_numbers=True, keep_censored=True, min_censor_days=30)
trcr_c = TRACERx(hgnc, cutoff=730, window=730, req_NT=True, window_positives=False, print_excl_numbers=True)

N primary LUAD patients from all sources combined (pre-criteria): 1212
Exclusion: Lost to follow-up/redacted (n=31) --> 1181 remaining
Exclusion: Missing smoking status (n=704) --> 477 remaining
Exclusion: Missing age (n=19) --> 458 remaining
Exclusion: No usable recurrence information (n=0) --> 458 remaining
Exclusion: No usable 730-day label (n=9) --> 449 remaining
Retained 167 censored patients (recurrence-free, follow-up < 730d): label_any is <NA>, fu_time/event_obs usable for a time-to-event loss
Exclusion: Missing sex (n=0) --> 449 remaining
Exclusion: Missing stage (n=2) --> 447 remaining
Exclusion: Stage IV (n=26) --> 421 remaining
Exclusion: Stage IIIB (n=7) --> 414 remaining
Exclusion: Pre-7th edition with T4 (n=1) --> 413 remaining
Exclusion: Missing pathologic N (n=0) --> 413 remaining
Exclusion: Pathologic NX (n=6) --> 407 remaining
Exclusion: Missing pathologic T (n=0) --> 407 remaining
Expression shape: (510, 17375), Mutation shape: (562, 15736)
All have clinical (for la

In [15]:
tcga_c.ttr_src.value_counts()

dist_metas_S                                         39
days_to_nte                                          31
local_rec_S                                          30
DFI.time_7                                           14
rec_RI                                               11
DFS_DAYS_3                                           10
dist_rec_TS                                           6
days_to_additional_surgery_metastatic_procedure_1     2
local_rec_TS|rec_RI                                   1
local_rec_TS                                          1
dist_newprim_S                                        1
local_dist_S                                          1
Name: count, dtype: Int64

In [16]:
# justification for binarization
for name, m in [('TCGA', tcga_c.mut), ('TRACERx', trcr_c.mut)]:
    nz = m.to_numpy()
    nz = nz[nz > 0]
    print(f'{name}: {len(nz)} nonzero entries, {(nz > 1).mean():.2%} with >1 mutation, {(nz > 2).mean():.2%} with >2 mutation, max {nz.max():.0f}\n')

tcga_c, trcr_c, conc_dict = format_for_modeling(tcga_c, trcr_c, mut_transform='binarize', use_NT=True, conc_min_count=0) 
lung_c = LUNG(tcga_c, trcr_c, 'data/')

TCGA: 95429 nonzero entries, 16.00% with >1 mutation, 1.43% with >2 mutation, max 17

TRACERx: 51304 nonzero entries, 6.51% with >1 mutation, 1.03% with >2 mutation, max 14

Overlapping genes: 11836; concordant: 8092; after FLAGS removal: 8012
(reporting only) mutated in >=0 patients in either cohort: 11836


  144 censored patients (label <NA>, usable for a time-to-event loss), 248 with determinate early-recurrence labels
Clin cols: 8  Expr cols: 17236  Mut cols: 8012


In [75]:
_ = [col_stats(ds.clin_for_table1, combined_col=True, valset_max=5, save=f'results/other/{ds.name}_feature_nulls.csv') for ds in [tcga_c, trcr_c]]

In [16]:
import harmonization_numbers; importlib.reload(harmonization_numbers)
from harmonization_numbers import get_mutation_hmzn_numbers
"""
pool_min_count=0 gates the feature pool: every usable gene faces the ratio test and prevalence is left to the training fold;
report_min_count=10 gates the published concordance statistics: below ~10 events the frequency ratio is pinned to integer multiples of n_tcga/n_trcr, so a median over rarer genes measures that lattice rather than the cohorts
"""
ds_kwargs = dict(cutoff=730, window=730, req_NT=True, window_positives=False, keep_censored=True, min_censor_days=30)
nums, tables = get_mutation_hmzn_numbers(ignore, hgnc, pool_min_count=0, report_min_count=10, composition_layout='wide', **ds_kwargs)

(TCGA-only arguments not passed to TRACERx: keep_censored, min_censor_days)
------------------- BEFORE HARMONIZATION -------------------
N primary LUAD patients from all sources combined (pre-criteria): 1212
Expression shape: (510, 17375), Mutation shape: (562, 16184)
All have clinical (for label):   Expr + Mut = 392   Expr only = 4   Mut only = 4   No genomic = 7
392 patients (147 relapsed, 101 disease-free, 144 censored)
392 patients (116 early, 132 late or never, 144 censored)

N primary LUAD patients from all sources combined (pre-criteria): 232
All have clinical (for label):   Expr + Mut = 150   Expr only = 0   Mut only = 0   No genomic = 0
150 patients (63 relapsed, 87 disease-free)
150 patients (53 early, 97 late or never)
Overlapping genes: 13992; concordant: 5663; after FLAGS removal: 5619
(reporting only) mutated in >=0 patients in either cohort: 13992
shift_table: 3890 of 13992 genes assessable at min_count=10
                               n_genes  median_freq_tcga  median_

In [38]:
tbl1 = Table1(tcga_c, trcr_c, clean_up=True)
tbl1.get_tables('other/tcga_table1.csv', 'other/tracerx_table1.csv')

In [35]:
for n, d in zip(['tcga_x', 'tcga_y', 'tracerx_x', 'tracerx_y', 'tcga_surv'], [lung_c.tcga_x, lung_c.tcga_y, lung_c.trcr_x, lung_c.trcr_y, lung_c.tcga_surv]):
    d.to_csv(f'data/processed_data/{n}.csv')

In [ ]:
import custom_transformers, fold_preprocessing, model, train_util, nested_cv, store_run
for m in [custom_transformers, fold_preprocessing, model, train_util, nested_cv, store_run]: importlib.reload(m)

from train_util import calibration_metrics, compute_metrics, bootstrap_metrics
from nested_cv import make_fixed_split, fixed_split_eval
from fold_preprocessing import fit_fold_pipeline, FOLD_DEFAULTS, MUT_RANKER_PARAMS
from model import PLASMA
from store_run import save_run, load_run, metrics_table, predictions, merge_results

import optuna

known_genes = prior.luad_genes

In [21]:
if 'device' not in globals(): 
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(device, torch.__version__)

def save_split(split, fn, lung, test_frac, seed, dp='data/'):
    write_file('pkl', fn, {'split': split, 'index': lung.tcga_x.index.tolist(), 'test_frac': test_frac, 'seed': seed}, dp)

def load_split(fn, lung, dp='data/'):
    d = load_data('pkl', fn, dp)
    assert d['index'] == lung.tcga_x.index.tolist(), 'cohort changed since split was saved'
    return d['split']

# split_c = make_fixed_split(lung_c.tcga_y, test_frac=0.2, seed=42)
# save_split(split_c, 'fixed_split_cens_20p_42', lung_c, 0.2, 42)
split_c = load_split('fixed_split_cens_20p_42', lung_c, dp='../data/')

def paired(a, b, col='te'):
    d = a[col].values - b[col].values
    p = wilcoxon(a[col], b[col]).pvalue if np.any(d != 0) else 1.0
    print(f'mean diff {d.mean():+.3f} | better on {(d > 0).sum()}/{len(d)} folds | Wilcoxon p={p:.2f}')

In [22]:
tr_idx, te_idx = split_c
x_tr, y_tr = lung_c.tcga_x.iloc[tr_idx], lung_c.tcga_y.iloc[tr_idx]
det = y_tr.notna().to_numpy() 

def gene_prior_sensitivity(verbose=False, **kw):
    f_w = fit_fold_pipeline(x_tr[det], y_tr[det], lung_c.cols, known_genes=known_genes, seed=42, verbose=verbose, **{**dict(scale_clin=True, expr_known_bonus=200, mut_boost_factor=3.5), **kw})
    if verbose: print()
    f_wo = fit_fold_pipeline(x_tr[det], y_tr[det], lung_c.cols, known_genes=known_genes, seed=42, verbose=verbose, **{**dict(scale_clin=True, expr_known_bonus=0, mut_boost_factor=1.0), **kw})
    if verbose: print()
    expr_added = [g.split('_')[0] for g in f_w['cols']['expr'] if g not in f_wo['cols']['expr']]
    mut_added = [g.split('_')[0] for g in f_w['cols']['mut'] if g not in f_wo['cols']['mut']]    
    print(f"expr added by prior: {expr_added}\nmut added by prior: {mut_added}")

In [23]:
gene_prior_sensitivity(verbose=True)

RemoveNearlyConstant - removed: 0 clin, 42 expr, 825 mut
RemoveRareMutations - 6530 mut removed (657 mut remaining, min_count=10)
RemoveLowExpression - 1476 expr removed (15718 expr remaining)
FilterVariance - 14146 expr removed, 1572 remaining (threshold=3.0774)
MiniCorrelationAnalysis - removed 3 expr (1569 remain), 0 mut (657 remain)
WeightedUnivar - 1519 expr removed, 50 expr remaining
SelectFeatures - mut selected: 50
  features: {'clin': 8, 'expr': 50, 'mut': 50}

RemoveNearlyConstant - removed: 0 clin, 42 expr, 825 mut
RemoveRareMutations - 6530 mut removed (657 mut remaining, min_count=10)
RemoveLowExpression - 1476 expr removed (15718 expr remaining)
FilterVariance - 14146 expr removed, 1572 remaining (threshold=3.0774)
MiniCorrelationAnalysis - removed 3 expr (1569 remain), 0 mut (657 remain)
WeightedUnivar - 1519 expr removed, 50 expr remaining
SelectFeatures - mut selected: 50
  features: {'clin': 8, 'expr': 50, 'mut': 50}

expr added by prior: ['CYP2E1', 'ENTPD2', 'NTRK2',

In [8]:
plasma_ = load_run('plasma_oct20', run_dir='../results/saved_runs', lung=lung_c)

In [27]:
# Search spaces equal in size
GRID = {'enc_lr': [3e-4, 6e-4], 'fus_lr': [1e-4, 3e-4], 'wd': [3e-4, 1e-3]} 

def make_space(version):
    def space(trial):
        wd = trial.suggest_categorical('wd', GRID['wd'])
        enc = dict(opt='adamw', lr=trial.suggest_categorical('enc_lr', GRID['enc_lr']), weight_decay=wd)
        fus = dict(opt='adamw', lr=trial.suggest_categorical('fus_lr', GRID['fus_lr']), weight_decay=wd)
        return {'clin_opt': enc, 'expr_opt': enc, 'mut_opt': enc, 'fus_opt': fus} 
    return space

SPACES = {v: make_space(v) for v in ('C', 'E', 'M', 'CE', 'CM', 'EM', 'CEM')}
n_trials = math.prod(len(v) for v in GRID.values())

best = dict(
    version = 'CEM',
    clin = dict(input_drop=0, n=[16], out=16, dropout=[0.1], act='gelu', norm=False),
    expr = dict(input_drop=0, n=[32, 16], out=16, dropout=[0.3, 0.1], act='gelu', norm=True),
    mut  = dict(input_drop=0, n=[32, 16], out=16, dropout=[0.5, 0.3], act='leaky_relu', norm=False),
    fus  = dict(embed=16, n=[16, 8], dropout=[0.1, 0], act='leaky_relu', mode='attention'),
    clin_skip = True,
    att = dict(use=True, n_heads=4, dropout=0.3, gate_input=True),
) 

fse_args = dict(n_seeds=5, k_fit=5, fixed_epochs=True, n_boot=1000, seed=42, known_genes=known_genes, fold_kwargs=dict(scale_clin=True, expr_known_bonus=200, mut_boost_factor=3.5))

""" 
plasma = fixed_split_eval(lung_c, device, split_c, best, versions=('CEM',), search_space=SPACES, sampler=lambda: optuna.samplers.GridSampler(GRID, seed=42), n_trials=n_trials, k_inner=3, return_models=True, **fse_args)
RemoveNearlyConstant - removed: 0 clin, 42 expr, 825 mut
RemoveRareMutations - 6530 mut removed (657 mut remaining, min_count=10)
RemoveLowExpression - 1476 expr removed (15718 expr remaining)
FilterVariance - 14146 expr removed, 1572 remaining (threshold=3.0774)
MiniCorrelationAnalysis - removed 3 expr (1569 remain), 0 mut (657 remain)
WeightedUnivar - 1519 expr removed, 50 expr remaining
SelectFeatures - mut selected: 50
  fold features: {'clin': 8, 'expr': 50, 'mut': 50}
fixed split: train 342 (198 labelled + 144 censored), test 50, TRACERx 150
  CEM  tuned over 8 trials: inner-CV AUC 0.730  {'wd': 0.0003, 'enc_lr': 0.0006, 'fus_lr': 0.0003}
  CEM  internal AUC 0.816 [0.690-0.920] | AUPRC 0.753 [0.545-0.922] | bal_acc 0.703 [0.567-0.825]
  CEM  external AUC 0.789 [0.714-0.857] | AUPRC 0.661 [0.519-0.784] | bal_acc 0.705 [0.627-0.781]
"""

best_hps = {**{f'{k}_opt': dict(opt='adamw', lr=6e-4, weight_decay=3e-4) for k in ['clin', 'expr', 'mut']}, 'fus_opt': dict(opt='adamw', lr=3e-4, weight_decay=3e-4)}
best = {**best, **best_hps}

plasma = fixed_split_eval(lung_c, device, split_c, best, versions=('CEM',), return_models=True, **fse_args)
display(metrics_table(plasma).round(3))

RemoveNearlyConstant - removed: 0 clin, 42 expr, 825 mut
RemoveRareMutations - 6530 mut removed (657 mut remaining, min_count=10)
RemoveLowExpression - 1476 expr removed (15718 expr remaining)
FilterVariance - 14146 expr removed, 1572 remaining (threshold=3.0774)
MiniCorrelationAnalysis - removed 3 expr (1569 remain), 0 mut (657 remain)
WeightedUnivar - 1519 expr removed, 50 expr remaining
SelectFeatures - mut selected: 50
  features: {'clin': 8, 'expr': 50, 'mut': 50}
fixed split: train 342 (198 labelled + 144 censored), test 50, TRACERx 150
  CEM  internal AUC 0.816 [0.690-0.920] | AUPRC 0.753 [0.545-0.922] | bal_acc 0.703 [0.567-0.825]
  CEM  external AUC 0.789 [0.714-0.857] | AUPRC 0.661 [0.519-0.784] | bal_acc 0.705 [0.627-0.781]


,version,split,auc,auprc,avg_prec,acc,bal_acc,prec,rec,spec,f1,tn,fp,fn,tp,brier,bss,cal_slope,cal_citl
0,CEM,oof,0.723,0.729,0.730,0.667,0.671,0.622,0.742,0.600,0.676,63,42,24,69,0.211,0.155,NaN,NaN
1,CEM,internal,0.816,0.753,0.766,0.700,0.703,0.654,0.739,0.667,0.694,18,9,6,17,0.188,0.245,1.829,-0.065
2,CEM,external,0.789,0.661,0.665,0.707,0.705,0.569,0.698,0.711,0.627,69,28,16,37,0.185,0.192,1.556,-0.309


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.ticker import FixedLocator

import shap_explain, shap, baselines, compare_variants, results_tables, plotting as P
for m in [shap_explain, baselines, compare_variants, results_tables, P]: importlib.reload(m)

from shap_explain import explain, top_features, modality_contribution, plot_beeswarm, italicize_gene_yticklabels, save_explanation, load_explanation
from baselines import baseline_fixed_split_eval
from compare_variants import paired_compare, combine_splits, _holm
from results_tables import performance_table, comparison_table, resolution_table

In [29]:
def calibration_report(res, model='PLASMA', splits=('internal', 'external'), nbins=5, n_sim=2000, seed=0):
    rows = []
    for split in splits:
        e = res[model]['ens'][split]
        y = np.asarray(e['targets']).astype(int).ravel()
        p = np.asarray(e['probs_cal'], dtype=float).ravel()
        m = calibration_metrics(y, p)
        prev = float(y.mean())
        ref = prev * (1 - prev)
        rng = np.random.default_rng(seed)
        sl = [s for s in (calibration_metrics(rng.binomial(1, p), p)['cal_slope'] for _ in range(n_sim)) if np.isfinite(s)]
        lo, hi = np.percentile(sl, [2.5, 97.5])
        edges = np.unique(np.quantile(p, np.linspace(0, 1, nbins + 1)))
        low = p <= edges[1]
        rows.append({'split': split, 'n': len(y), 'prevalence': round(prev, 3), 'brier': round(m['brier'], 3), 'brier_ref': round(ref, 3), 'brier_skill': round(1 - m['brier'] / ref, 3),
                     'cal_slope': round(m['cal_slope'], 2), 'slope_null_lo': round(lo, 2), 'slope_null_hi': round(hi, 2), 'slope_in_band': bool(lo <= m['cal_slope'] <= hi),
                     'cal_citl': round(m['cal_citl'], 2), 'lowest_bin': f'{int(y[low].sum())}/{int(low.sum())}'})
    return pd.DataFrame(rows)

rep = calibration_report({'PLASMA': plasma['CEM']})
print(rep.to_string(index=False))
rep.to_csv('results/other/plasma_calibration.csv', index=False)

   split   n  prevalence  brier  brier_ref  brier_skill  cal_slope  slope_null_lo  slope_null_hi  slope_in_band  cal_citl lowest_bin
internal  50       0.460  0.188      0.248        0.245       1.83           0.26           2.21           True     -0.07       1/10
external 150       0.353  0.185      0.228        0.192       1.56           0.56           1.58           True     -0.31       1/30


In [ ]:
entry = plasma['CEM']

# ex = explain(entry, device, background='train', explain_on='test', n_background=150, seed=42)
# save_explanation(ex, 'results/shap_results')
# top_shap = top_features(ex, n=20).round(4)
# top_shap.to_csv('results/shap_results/top20_shap.csv', index=False)

ex = load_explanation('results/shap_results')
top_shap = top_features(ex, n=20).round(4)

print(top_shap.to_string(index=False))
print(modality_contribution(ex).round(4).to_string(index=False))

       feature  mean_abs_shap  mean_shap  corr_with_shap
        path_T         0.2526    -0.0211          0.9855
         stage         0.1876     0.0008          0.9716
    TEX15_expr         0.1795     0.0331          0.9748
current_smoker         0.1647    -0.0119         -0.9789
           age         0.1622    -0.0320          0.9712
   CALML5_expr         0.1545     0.0447          0.9738
   ZNF486_expr         0.1412    -0.0229         -0.9611
    WDR87_expr         0.1244    -0.0317         -0.9449
  never_smoker         0.0967     0.0055         -0.9697
    CCL20_expr         0.0961    -0.0012          0.9456
  ANGPTL4_expr         0.0887     0.0245          0.9470
     DKK1_expr         0.0841    -0.0089          0.9686
  CYP21A2_expr         0.0828     0.0319          0.9605
      HAL_expr         0.0793     0.0252          0.9546
      CA4_expr         0.0781     0.0096         -0.9306
    KRT81_expr         0.0775    -0.0071          0.9742
  TMEM213_expr         0.0772  

In [ ]:
det = lung_c.tcga_y.notna()
X, y = lung_c.tcga_x[det], lung_c.tcga_y[det].astype(int).to_numpy()

yt_all = pd.to_numeric(lung_c.trcr_y, errors='coerce')
det_t = yt_all.notna()
yt = yt_all[det_t].astype(int).to_numpy()
Xt = lung_c.trcr_x[det_t]
st_tcga = tcga_c.clin_for_table1.loc[X.index, 'smoking_status'].replace({'ref_sub15': 'former', 'ref_sup15': 'former', 'ref_unk': 'former'})
st_trcr = trcr_c.clin_for_table1.loc[Xt.index, 'smoking_status'].replace({'ref': 'former'})
print(len(X), len(y), '|', int(det_t.sum()), len(yt))
print(st_tcga.value_counts().to_dict(), st_trcr.value_counts().to_dict())

def unadj_or(st, yy, a='current', b='former'):
    st = pd.Series(np.asarray(st)).reset_index(drop=True); yy = np.asarray(yy).astype(int)
    ma, mb = (st == a).to_numpy(), (st == b).to_numpy()
    t = np.array([[int(yy[ma].sum()), int((1 - yy[ma]).sum())], [int(yy[mb].sum()), int((1 - yy[mb]).sum())]])
    p = fisher_exact(t)[1]
    if (t == 0).any():
        warnings.warn(f'zero cell in {a} vs {b} table {t.tolist()}; Haldane-Anscombe 0.5 applied to the odds ratio only')
        c = t + 0.5
    else:
        c = t.astype(float)
    (n11, n10), (n01, n00) = c
    lo_ = np.log((n11 * n00) / (n10 * n01)); se = np.sqrt(1/n11 + 1/n10 + 1/n01 + 1/n00)
    return (np.exp(lo_), np.exp(lo_ - 1.96*se), np.exp(lo_ + 1.96*se)), p

""" OR, 95% CI and P for term, all out of one likelihood """
def or_from_logit(Xc, yc, covars=('stage', 'age', 'sex'), term='current_smoker'):
    d = Xc[['current_smoker', 'never_smoker', *covars]].astype(float)
    m = sm.Logit(np.asarray(yc).astype(int), sm.add_constant(d)).fit(disp=0)
    b, se, p = m.params[term], m.bse[term], m.pvalues[term]
    return (np.exp(b), np.exp(b - 1.96 * se), np.exp(b + 1.96 * se)), float(p)

""" Two-proportion score test = Pearson chi-square on the 2x2 (z**2 == chi2); tests equality of the observed proportions """
def prop_test(st, yy, a='current', b='former'):
    st = pd.Series(np.asarray(st)).reset_index(drop=True); yy = np.asarray(yy).astype(int)
    ma, mb = (st == a).to_numpy(), (st == b).to_numpy()
    t = np.array([[int(yy[ma].sum()), int((1 - yy[ma]).sum())], [int(yy[mb].sum()), int((1 - yy[mb]).sum())]])
    _, p, *_ = chi2_contingency(t, correction=False)
    return t, float(p)

""" Current vs. former smoker for one binary outcome: observed rates and score-test P (panel brackets), unadjusted and adjusted ORs from logistic fits, and inverse-variance pooled estimates """
def smoking_analysis(y_tcga, y_trcr, outcome):
    cohorts = {'TCGA': (X, y_tcga, st_tcga), 'TRACERx': (Xt, y_trcr, st_trcr)}
    unadj, adj, tests, smoke = {}, {}, {}, {}
    print(f'── {outcome} ──')
    for nm, (Xc, yc, st) in cohorts.items():
        unadj[nm], p_wald = or_from_logit(Xc, yc, covars=())
        adj[nm], _ = or_from_logit(Xc, yc)
        t, p_score = prop_test(st, yc)
        (o22, _, _), p_fisher = unadj_or(st, yc)
        assert abs(o22 - unadj[nm][0]) < 1e-6, f'{nm}: 2x2 OR {o22:.6f} != saturated-logit OR {unadj[nm][0]:.6f} — the current/never dummies and the collapsed smoking_status labels disagree about "former"'
        (a, b), (c, d) = t
        smoke[nm] = dict(table=t.tolist(), p_score=p_score, p_wald=p_wald, p_fisher=p_fisher, rate_cur=a/(a+b), rate_for=c/(c+d))
        tests[nm] = ('current', 'former', p_score)
        print(f'{nm:9} current {a}/{a+b} ({a/(a+b):.1%})  former {c}/{c+d} ({c/(c+d):.1%})   score P = {p_score:.4f}   Wald P = {p_wald:.4f}   Fisher P = {p_fisher:.4f}')
    *pooled, p_pooled = P.pool_log_or(unadj)
    *pooled_adj, p_pooled_adj = P.pool_log_or(adj)
    est = {'TCGA': unadj['TCGA'], 'TCGA*': adj['TCGA'], 'TRACERx': unadj['TRACERx'], 'TRACERx*': adj['TRACERx'], 'Pooled': tuple(pooled), 'Pooled*': tuple(pooled_adj)}
    for lab, (o, lo, hi) in est.items():
        print(f'{lab:10} OR {o:.2f} [{lo:.2f}-{hi:.2f}]')
    print(f'pooled P = {p_pooled:.4f}, pooled adjusted P = {p_pooled_adj:.4f}\n')
    return dict(est=est, tests=tests, smoke=smoke, p_pooled=p_pooled, p_pooled_adj=p_pooled_adj)

early = smoking_analysis(y, yt, 'early recurrence')
late  = smoking_analysis(1 - y, 1 - yt, 'late or no recurrence')

# every late/never OR should be the reciprocal of the early one (same patients, outcome flipped)
for k in early['est']:
    assert abs(early['est'][k][0] * late['est'][k][0] - 1) < 1e-4, f'{k}: {early["est"][k][0]:.4f} x {late["est"][k][0]:.4f} != 1'

248 248 | 150 150
{'former': 150, 'current': 62, 'never': 36} {'former': 73, 'current': 60, 'never': 17}
── early recurrence ──
TCGA      current 21/62 (33.9%)  former 80/150 (53.3%)   score P = 0.0099   Wald P = 0.0107   Fisher P = 0.0106
TRACERx   current 17/60 (28.3%)  former 33/73 (45.2%)   score P = 0.0456   Wald P = 0.0472   Fisher P = 0.0501
TCGA       OR 0.45 [0.24-0.83]
TCGA*      OR 0.49 [0.25-0.94]
TRACERx    OR 0.48 [0.23-0.99]
TRACERx*   OR 0.49 [0.20-1.24]
Pooled     OR 0.46 [0.29-0.74]
Pooled*    OR 0.49 [0.29-0.83]
pooled P = 0.0012, pooled adjusted P = 0.0086

── late or no recurrence ──
TCGA      current 41/62 (66.1%)  former 70/150 (46.7%)   score P = 0.0099   Wald P = 0.0107   Fisher P = 0.0106
TRACERx   current 43/60 (71.7%)  former 40/73 (54.8%)   score P = 0.0456   Wald P = 0.0472   Fisher P = 0.0501
TCGA       OR 2.23 [1.21-4.13]
TCGA*      OR 2.06 [1.07-3.95]
TRACERx    OR 2.09 [1.01-4.31]
TRACERx*   OR 2.03 [0.80-5.13]
Pooled     OR 2.17 [1.36-3.47]
Pooled*   

In [ ]:
fig = plt.figure(figsize=(11, 6.5), dpi=1200)
gs = GridSpec(2, 2, figure=fig, height_ratios=[1.4, 1], hspace=.3, wspace=.2, width_ratios=[0.55, 0.42])
ax_shap, ax_b, ax_c = fig.add_subplot(gs[:, 0]), fig.add_subplot(gs[0, 1]), fig.add_subplot(gs[1, 1])

plot_beeswarm(ex, n=20, arrow_y=.02, frac=.5, base_frac=.03, bottom_pad=0.25, title='SHAP feature importance', ax=ax_shap)
italicize_gene_yticklabels(ax_shap)

P.plot_observed_association(dict(TCGA=(st_tcga, 1 - y), TRACERx=(st_trcr, 1 - yt)),
                            order=['current', 'former', 'never'], labels=dict(current='Current', former='Former', never='Never'),
                            bracket_gap=0.045, bracket_step=0.09, annotate_n=False, headroom=0.12, tests=late['tests'],
                            legend_kw=dict(loc='upper right', ncol=2, fontsize=7, frameon=True, framealpha=.85, edgecolor='0.7', borderpad=.35, columnspacing=0.8, handlelength=1.5, handletextpad=0.5),
                            ylabel='Late/no recurrence', title='Observed late/no recurrence by smoking status', ax=ax_b)
ax_b.set_ylim(0, max(ax_b.get_ylim()[1], 1.0))
ax_b.yaxis.set_major_locator(FixedLocator(np.arange(0, 1.0 + 1e-9, 0.2)))

P.plot_or_forest(late['est'], title='Late/no recurrence: current vs. former smoker', ax=ax_c, legend=True,
                 legend_kw=dict(loc='lower right', ncol=1, fontsize=6.5, markerscale=0.5, frameon=True, framealpha=.85, edgecolor='0.7', borderpad=.35, columnspacing=0.8, handlelength=1.1, handletextpad=0.5),
                 legend_labels={'pooled': 'Unadjusted', 'pooled, adjusted': 'Adjusted', 'unadjusted': 'Unadjusted', 'adjusted': 'Adjusted'},
                 note='* adjusted for age, stage, and sex')

P.panel_labels([ax_shap, ax_b, ax_c], 'ABC')
plt.show()
fig.savefig(f'results/figures/shap_smoking_late_never.png', bbox_inches='tight', dpi=1200)

/oscar/data/dgamsiz/jpatrico/LUNG4/shap_explain.py:239: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(phi, X, feature_names=[clean_feature(f) for f in names], show=False, max_display=n, plot_size=None, **kwargs)
/oscar/home/jpatrico/torch_venv/lib64/python3.9/site-packages/shap/plots/_beeswarm.py:1150: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


In [ ]:
def backfill_metrics(res, keys=('bss',), n_boot=1000, seed=0, splits=None):
    """ Recompute stored point estimates and CIs from saved predictions, for metrics added to compute_metrics after the run that produced res. Mutates res in place """
    for m, r in res.items():
        thr = r.get('threshold')
        if thr is None:
            if any(k in ('bal_acc', 'prec', 'rec', 'spec', 'f1', 'acc', 'tp', 'fp', 'fn', 'tn') for k in keys):
                continue
            thr = 0.5
        for split, e in (r.get('ens') or {}).items():
            if splits and split not in splits:
                continue
            y, p = e.get('targets'), e.get('probs_cal')
            if y is None or p is None:
                continue
            pt = compute_metrics(y, p, threshold=thr, plot_metrics=False)
            r.setdefault('metrics', {}).setdefault(split, {}).update({k: pt[k] for k in keys if k in pt})
            r.setdefault('ci', {}).setdefault(split, {}).update(bootstrap_metrics(y, p, threshold=thr, n_boot=n_boot, seed=seed, keys=tuple(keys)))
    return res

#### **BASELINES**

In [ ]:
ptable_path = 'results/performance_tables'
ctable_path = 'results/comparison_tables'
rtable_path = 'results/resolution_tables'
ptable_metrics = ('auc', 'auprc', 'avg_prec', 'brier', 'bss', 'bal_acc', 'rec', 'spec', 'prec', 'cal_slope', 'cal_citl')

def run_comparison(res, ptable_prefix, ctable_prefix, ref='PLASMA', secondary_metric='auprc', res_tables=True, order=None, counts=False):
    pt = performance_table(res, ['internal', 'external'], metrics=ptable_metrics, out=f'{ptable_path}/{ptable_prefix}', order=order, counts=counts)
    cmp_ = paired_compare(res, ref=ref, metrics=('auc', secondary_metric), n_boot=5000, n_perm=10000, seed=42)
    pooled = {m: combine_splits(cmp_, metric=m, splits=('internal', 'external')) for m in ('auc', secondary_metric)}
    for t in pooled.values(): t['p_holm'] = _holm(t['p'].to_numpy())
    if order is not None: order.remove(ref)
    c1 = comparison_table(cmp_, pooled['auc'], metric='auc', ref=ref, show_q=True, delong=True, cohort_p=True, show_raw_p=True, power=True, out=f'{ctable_path}/{ctable_prefix}_auc', order=order)
    r1 = resolution_table(cmp_, pooled['auc'], metric='auc', out=f'{rtable_path}/{ctable_prefix}_auc')
    c2 = comparison_table(cmp_, pooled[secondary_metric], metric=secondary_metric, ref='PLASMA', show_q=True, show_raw_p=True, power=True, out=f'{ctable_path}/{ctable_prefix}_{secondary_metric}', order=order)
    r2 = resolution_table(cmp_, pooled[secondary_metric], metric=secondary_metric, out=f'{rtable_path}/{ctable_prefix}_{secondary_metric}')

In [18]:
bl = baseline_fixed_split_eval(lung_c, split_c, names=('LR', 'RF', 'MLP', 'SVM', 'KNN'), known_genes=known_genes, fold_kwargs=dict(scale_clin=True, expr_known_bonus=200, mut_boost_factor=3.5))

RemoveNearlyConstant - removed: 0 clin, 42 expr, 825 mut
RemoveRareMutations - 6530 mut removed (657 mut remaining, min_count=10)
RemoveLowExpression - 1476 expr removed (15718 expr remaining)
FilterVariance - 14146 expr removed, 1572 remaining (threshold=3.0774)
MiniCorrelationAnalysis - removed 3 expr (1569 remain), 0 mut (657 remain)
WeightedUnivar - 1519 expr removed, 50 expr remaining
SelectFeatures - mut selected: 50
  fold features: {'clin': 8, 'expr': 50, 'mut': 50}
baselines [clin+expr+mut]: train 198 labelled (censored patients cannot be used by a binary classifier), test 50, TRACERx 150, 108 features
  LR   grid   8 pts, inner-CV AUC 0.810  {'C': 0.1, 'penalty': 'l2'}
  LR         internal AUC 0.725 | external 0.682 | oof 0.788
  RF   grid   9 pts, inner-CV AUC 0.729  {'max_depth': None, 'min_samples_leaf': 1}
  RF         internal AUC 0.694 | external 0.712 | oof 0.715
  MLP  grid   9 pts, inner-CV AUC 0.818  {'lr0': 0.001, 'hidden_layer_sizes': '(128,64)'}
  MLP        int

In [19]:
plasma_bl = merge_results({'PLASMA': plasma['CEM']}, bl)
run_comparison(plasma_bl, 'blines', 'plasma_v_blines', ref='PLASMA', secondary_metric='bss', order=['PLASMA', 'MLP', 'SVM', 'LR', 'KNN', 'RF'])

TCGA test (n = 50) The no-information Brier score for this set, obtained by predicting its event rate for every patient, is 0.248.
TRACERx (n = 150) The no-information Brier score for this set, obtained by predicting its event rate for every patient, is 0.228.


,Cohort,Model,AUROC,AUPRC,Avg prec,Brier,BSS,Bal acc,Recall,Spec,Prec,Cal slope,Cal CIL
0,TCGA test,PLASMA,0.816 (0.690-0.920),0.753 (0.545-0.922),0.766 (0.584-0.925),0.188 (0.153-0.225),0.25 (0.07-0.37),0.70 (0.57-0.82),0.74 (0.53-0.90),0.67 (0.48-0.84),0.65 (0.46-0.82),1.83 (0.93-3.51),-0.07 (-0.63 to 0.44)
1,TCGA test,MLP,0.734 (0.588-0.862),0.645 (0.440-0.870),0.671 (0.482-0.875),0.208 (0.157-0.264),0.16 (-0.09 to 0.36),0.66 (0.53-0.78),0.70 (0.50-0.88),0.63 (0.46-0.80),0.62 (0.42-0.79),0.89 (0.35-1.66),-0.16 (-0.82 to 0.47)
2,TCGA test,SVM,0.728 (0.585-0.862),0.685 (0.466-0.878),0.699 (0.506-0.883),0.211 (0.160-0.265),0.15 (-0.11 to 0.34),0.64 (0.50-0.77),0.61 (0.40-0.80),0.67 (0.50-0.83),0.61 (0.39-0.80),0.87 (0.33-1.69),-0.14 (-0.80 to 0.47)
3,TCGA test,LR,0.725 (0.573-0.860),0.662 (0.447-0.849),0.674 (0.483-0.857),0.208 (0.162-0.256),0.16 (-0.07 to 0.34),0.69 (0.56-0.83),0.61 (0.40-0.80),0.78 (0.62-0.92),0.70 (0.48-0.89),0.96 (0.40-1.90),-0.15 (-0.78 to 0.44)
4,TCGA test,KNN,0.720 (0.559-0.861),0.610 (0.423-0.852),0.637 (0.462-0.861),0.216 (0.174-0.261),0.13 (-0.09 to 0.28),0.72 (0.59-0.84),0.70 (0.50-0.88),0.74 (0.58-0.89),0.70 (0.50-0.87),1.09 (0.28-2.51),-0.19 (-0.76 to 0.38)
5,TCGA test,RF,0.694 (0.538-0.828),0.589 (0.398-0.822),0.616 (0.432-0.835),0.220 (0.178-0.265),0.11 (-0.13 to 0.27),0.61 (0.47-0.74),0.43 (0.24-0.65),0.78 (0.62-0.92),0.62 (0.37-0.84),0.94 (0.21-1.97),-0.10 (-0.72 to 0.49)
6,TRACERx,PLASMA,0.789 (0.714-0.857),0.661 (0.519-0.784),0.665 (0.536-0.788),0.185 (0.163-0.207),0.19 (0.07-0.28),0.70 (0.63-0.78),0.70 (0.57-0.81),0.71 (0.61-0.80),0.57 (0.45-0.70),1.56 (1.08-2.24),-0.31 (-0.67 to 0.03)
7,TRACERx,MLP,0.695 (0.608-0.780),0.582 (0.440-0.716),0.588 (0.456-0.721),0.203 (0.171-0.236),0.11 (-0.04 to 0.24),0.63 (0.56-0.71),0.47 (0.35-0.60),0.79 (0.71-0.87),0.56 (0.42-0.70),0.75 (0.42-1.12),-0.14 (-0.54 to 0.27)
8,TRACERx,SVM,0.723 (0.637-0.802),0.612 (0.474-0.736),0.617 (0.489-0.741),0.196 (0.164-0.229),0.14 (-0.00 to 0.26),0.66 (0.58-0.73),0.47 (0.34-0.60),0.85 (0.77-0.91),0.62 (0.48-0.77),0.85 (0.52-1.24),-0.12 (-0.52 to 0.28)
9,TRACERx,LR,0.682 (0.592-0.767),0.571 (0.430-0.702),0.576 (0.450-0.709),0.208 (0.172-0.244),0.09 (-0.04 to 0.21),0.62 (0.55-0.68),0.32 (0.20-0.44),0.92 (0.86-0.97),0.68 (0.50-0.86),0.67 (0.37-1.06),0.10 (-0.31 to 0.50)


,Model,TCGA test Δ (95% CI),TCGA test perm. P,TCGA test perm. P (Holm),TCGA test DeLong P,TCGA test DeLong P (Holm),TRACERx Δ (95% CI),TRACERx perm. P,TRACERx perm. P (Holm),TRACERx DeLong P,TRACERx DeLong P (Holm),Pooled Δ (95% CI),Pooled P,Pooled P (Holm),Het. P
0,MLP,0.082 (-0.002 to 0.174),0.107,0.283,0.061,0.175,0.094 (0.042-0.150),0.012,0.048,< 0.001,0.002,0.091 (0.045-0.137),< 0.001,< 0.001,0.816
1,SVM,0.089 (0.011-0.182),0.071,0.283,0.040,0.160,0.066 (0.020-0.113),0.035,0.048,0.004,0.011,0.071 (0.031-0.112),< 0.001,0.002,0.655
2,LR,0.092 (-0.002 to 0.198),0.083,0.283,0.067,0.175,0.108 (0.050-0.166),0.015,0.048,< 0.001,0.001,0.104 (0.054-0.154),< 0.001,< 0.001,0.789
3,KNN,0.097 (-0.002 to 0.205),0.103,0.283,0.058,0.175,0.081 (0.019-0.144),0.008,0.039,0.011,0.023,0.085 (0.032-0.139),0.002,0.003,0.805
4,RF,0.122 (0.023-0.234),0.023,0.115,0.018,0.092,0.077 (0.009-0.143),0.023,0.048,0.024,0.024,0.090 (0.034-0.147),0.002,0.003,0.471


,Model,TCGA test Δ (95% CI),TRACERx Δ (95% CI),Pooled Δ (95% CI),Pooled P,Pooled P (Holm),Het. P
0,MLP,0.083 (-0.039 to 0.223),0.081 (-0.002 to 0.162),0.082 (0.012-0.152),0.022,0.046,0.987
1,SVM,0.096 (-0.024 to 0.236),0.051 (-0.026 to 0.124),0.062 (-0.003 to 0.126),0.062,0.062,0.557
2,LR,0.082 (-0.046 to 0.225),0.101 (0.004-0.190),0.095 (0.018-0.172),0.015,0.046,0.814
3,KNN,0.114 (0.004-0.241),0.086 (0.021-0.152),0.093 (0.035-0.151),0.002,0.007,0.690
4,RF,0.133 (0.024-0.261),0.093 (0.018-0.167),0.104 (0.042-0.167),< 0.001,0.005,0.579


In [64]:
line_props = {
    'PLASMA': {'color': '#FC1C7D', 'lw': 2, 'solid_joinstyle': 'round'},
    'LR':  {'color': '#02B902', 'lw': 2, 'solid_joinstyle': 'round'},
    'RF':  {'color': '#961C04', 'lw': 2, 'solid_joinstyle': 'round'},
    'MLP': {'color': '#0073E6', 'lw': 2, 'solid_joinstyle': 'round'},
    'SVM': {'color': '#6600FF', 'lw': 2, 'solid_joinstyle': 'round'},
    'KNN': {'color': '#FFD700', 'lw': 2, 'solid_joinstyle': 'round'},
}
_ = P.plot_roc_pr(plasma_bl, house='PLASMA', line_props=line_props, fsize=(8, 8), hspace=.25, dpi=1200, label_panels=True, save='results/figures/roc_pr_oct20.png')

In [21]:
clin_ladder = {}
for sfx, fcols in [('-stage', ['stage']), ('-sas', ['stage', 'age', 'sex']), ('-clin', None)]:
    r = baseline_fixed_split_eval(
        lung_c, split_c, names=('LR',), known_genes=known_genes,
        fold_kwargs=dict(scale_clin=True, expr_known_bonus=200, mut_boost_factor=3.5),
        blocks=('clin',), suffix=sfx, feature_cols=fcols)
    clin_ladder.update(r)

RemoveNearlyConstant - removed: 0 clin, 42 expr, 825 mut
RemoveRareMutations - 6530 mut removed (657 mut remaining, min_count=10)
RemoveLowExpression - 1476 expr removed (15718 expr remaining)
FilterVariance - 14146 expr removed, 1572 remaining (threshold=3.0774)
MiniCorrelationAnalysis - removed 3 expr (1569 remain), 0 mut (657 remain)
WeightedUnivar - 1519 expr removed, 50 expr remaining
SelectFeatures - mut selected: 50
  fold features: {'clin': 8, 'expr': 50, 'mut': 50}
baselines [clin]: train 198 labelled (censored patients cannot be used by a binary classifier), test 50, TRACERx 150, 1 features
  LR   grid   8 pts, inner-CV AUC 0.577 (6 tied)  {'C': 0.01, 'penalty': 'l2'}
  LR-stage   internal AUC 0.545 | external 0.750 | oof 0.569
RemoveNearlyConstant - removed: 0 clin, 42 expr, 825 mut
RemoveRareMutations - 6530 mut removed (657 mut remaining, min_count=10)
RemoveLowExpression - 1476 expr removed (15718 expr remaining)
FilterVariance - 14146 expr removed, 1572 remaining (thresh

In [22]:
plasma_clin = merge_results({'PLASMA': plasma['CEM']}, clin_ladder)

In [23]:
run_comparison(plasma_clin, 'clin_ladder', 'plasma_v_clin_ladder', ref='PLASMA', secondary_metric='bss', order=['PLASMA', 'LR-clin', 'LR-sas', 'LR-stage'])

TCGA test (n = 50) The no-information Brier score for this set, obtained by predicting its event rate for every patient, is 0.248.
TRACERx (n = 150) The no-information Brier score for this set, obtained by predicting its event rate for every patient, is 0.228.


,Cohort,Model,AUROC,AUPRC,Avg prec,Brier,BSS,Bal acc,Recall,Spec,Prec,Cal slope,Cal CIL
0,TCGA test,PLASMA,0.816 (0.690-0.920),0.753 (0.545-0.922),0.766 (0.584-0.925),0.188 (0.153-0.225),0.25 (0.07-0.37),0.70 (0.57-0.82),0.74 (0.53-0.90),0.67 (0.48-0.84),0.65 (0.46-0.82),1.83 (0.93-3.51),-0.07 (-0.63 to 0.44)
1,TCGA test,LR-clin,0.750 (0.611-0.885),0.757 (0.572-0.896),0.762 (0.601-0.899),0.223 (0.205-0.242),0.10 (-0.01 to 0.15),0.62 (0.51-0.73),0.35 (0.15-0.56),0.89 (0.77-1.00),0.73 (0.44-1.00),4.61 (2.40-9.06),0.03 (-0.52 to 0.55)
2,TCGA test,LR-sas,0.571 (0.396-0.744),0.558 (0.357-0.767),0.574 (0.394-0.777),0.244 (0.232-0.256),0.02 (-0.09 to 0.04),0.45 (0.32-0.58),0.35 (0.15-0.55),0.56 (0.38-0.74),0.40 (0.19-0.61),3.82 (-2.05 to 10.69),-0.02 (-0.60 to 0.50)
3,TCGA test,LR-stage,0.545 (0.400-0.705),0.514 (0.315-0.739),0.480 (0.338-0.666),0.248 (0.229-0.265),0.00 (-0.13 to 0.06),0.56 (0.41-0.70),0.52 (0.32-0.74),0.59 (0.41-0.77),0.52 (0.31-0.74),0.59 (-2.01 to 3.86),-0.03 (-0.61 to 0.53)
4,TRACERx,PLASMA,0.789 (0.714-0.857),0.661 (0.519-0.784),0.665 (0.536-0.788),0.185 (0.163-0.207),0.19 (0.07-0.28),0.70 (0.63-0.78),0.70 (0.57-0.81),0.71 (0.61-0.80),0.57 (0.45-0.70),1.56 (1.08-2.24),-0.31 (-0.67 to 0.03)
5,TRACERx,LR-clin,0.743 (0.655-0.826),0.587 (0.440-0.731),0.595 (0.458-0.739),0.218 (0.206-0.229),0.05 (-0.08 to 0.11),0.71 (0.63-0.78),0.66 (0.53-0.79),0.75 (0.66-0.84),0.59 (0.46-0.72),2.99 (1.74-4.68),-0.49 (-0.82 to -0.16)
6,TRACERx,LR-sas,0.757 (0.669-0.836),0.615 (0.468-0.749),0.620 (0.484-0.756),0.232 (0.227-0.238),-0.02 (-0.14 to 0.04),0.70 (0.62-0.76),0.77 (0.65-0.88),0.62 (0.52-0.72),0.53 (0.41-0.64),8.20 (5.14-12.58),-0.51 (-0.84 to -0.18)
7,TRACERx,LR-stage,0.750 (0.670-0.827),0.683 (0.563-0.790),0.568 (0.445-0.695),0.220 (0.211-0.229),0.04 (-0.09 to 0.10),0.72 (0.64-0.79),0.77 (0.65-0.88),0.66 (0.57-0.75),0.55 (0.43-0.67),3.91 (2.55-5.70),-0.52 (-0.86 to -0.20)


,Model,TCGA test Δ (95% CI),TCGA test perm. P,TCGA test perm. P (Holm),TCGA test DeLong P,TCGA test DeLong P (Holm),TRACERx Δ (95% CI),TRACERx perm. P,TRACERx perm. P (Holm),TRACERx DeLong P,TRACERx DeLong P (Holm),Pooled Δ (95% CI),Pooled P,Pooled P (Holm),Het. P
0,LR-clin,0.066 (-0.076 to 0.208),0.380,0.380,0.352,0.352,0.046 (-0.026 to 0.122),0.285,0.856,0.223,0.669,0.051 (-0.015 to 0.116),0.132,0.166,0.809
1,LR-sas,0.246 (0.056-0.435),0.014,0.028,0.010,0.020,0.032 (-0.050 to 0.116),0.556,0.941,0.451,0.745,0.068 (-0.009 to 0.145),0.083,0.166,0.043
2,LR-stage,0.271 (0.093-0.440),0.004,0.013,0.002,0.007,0.039 (-0.047 to 0.126),0.470,0.941,0.373,0.745,0.085 (0.007-0.164),0.032,0.097,0.020


,Model,TCGA test Δ (95% CI),TRACERx Δ (95% CI),Pooled Δ (95% CI),Pooled P,Pooled P (Holm),Het. P
0,LR-clin,0.144 (0.001-0.281),0.145 (0.065-0.231),0.145 (0.074-0.215),< 0.001,< 0.001,0.992
1,LR-sas,0.229 (0.077-0.381),0.208 (0.114-0.309),0.214 (0.132-0.296),< 0.001,< 0.001,0.824
2,LR-stage,0.243 (0.091-0.398),0.155 (0.061-0.256),0.180 (0.098-0.262),< 0.001,< 0.001,0.341


In [67]:
fus_abl = {}
for fus_key in ['gated', 'concat']:
    fus_p = deepcopy(best)
    fus_p['fus']['mode'] = fus_key
    fus_abl[fus_key] = fixed_split_eval(lung_c, device, split_c, fus_p, versions=('CEM',), search_space=SPACES, sampler=lambda: optuna.samplers.GridSampler(GRID, seed=42), n_trials=n_trials, 
    k_inner=3, return_models=True, **fse_args)
    save_run(fus_abl[fus_key], f'{fus_key}_fus_oct20', kind='fixed_split', base=fus_p, fold_kwargs=dict(scale_clin=True, expr_known_bonus=200, mut_boost_factor=3.5), split=split_c,
         lung=lung_c, known_genes=known_genes, grid=GRID, notes=f'{fus_key} fusion')

RemoveNearlyConstant - removed: 0 clin, 42 expr, 825 mut
RemoveRareMutations - 6530 mut removed (657 mut remaining, min_count=10)
RemoveLowExpression - 1476 expr removed (15718 expr remaining)
FilterVariance - 14146 expr removed, 1572 remaining (threshold=3.0774)
MiniCorrelationAnalysis - removed 3 expr (1569 remain), 0 mut (657 remain)
WeightedUnivar - 1519 expr removed, 50 expr remaining
SelectFeatures - mut selected: 50
  fold features: {'clin': 8, 'expr': 50, 'mut': 50}
fixed split: train 342 (198 labelled + 144 censored), test 50, TRACERx 150
    inner best AUC 0.6399  params {'wd': 0.0003, 'enc_lr': 0.0006, 'fus_lr': 0.0003}
  CEM  tuned over 8 trials: inner-CV AUC 0.640  {'wd': 0.0003, 'enc_lr': 0.0006, 'fus_lr': 0.0003}
  CEM  internal AUC 0.758 [0.615-0.878] | AUPRC 0.691 [0.496-0.900] | bal_acc 0.607 [0.473-0.740]
  CEM  external AUC 0.782 [0.708-0.850] | AUPRC 0.636 [0.496-0.765] | bal_acc 0.704 [0.628-0.780]
  CEM  inner-CV 0.640 vs test 0.758 (gap -0.119: how optimistic th

In [24]:
fus_abl = {}
hps = {'gated': {'wd': 0.0003, 'enc_lr': 0.0006, 'fus_lr': 0.0003}, 'concat': {'wd': 0.0003, 'enc_lr': 0.0006, 'fus_lr': 0.0001}}

for fus_key in ['gated', 'concat']:
    hps_ = hps[fus_key]
    fus_p = deepcopy(best)
    fus_p['fus']['mode'] = fus_key
    best_hps = {**{f'{k}_opt': dict(opt='adamw', lr=hps_['enc_lr'], weight_decay=hps_['wd']) for k in ['clin', 'expr', 'mut']}, 'fus_opt': dict(opt='adamw', lr=hps_['fus_lr'], weight_decay=hps_['wd'])}
    fus_p = {**fus_p, **best_hps}
    fus_abl[fus_key] = fixed_split_eval(lung_c, device, split_c, fus_p, versions=('CEM',), return_models=True, **fse_args)

RemoveNearlyConstant - removed: 0 clin, 42 expr, 825 mut
RemoveRareMutations - 6530 mut removed (657 mut remaining, min_count=10)
RemoveLowExpression - 1476 expr removed (15718 expr remaining)
FilterVariance - 14146 expr removed, 1572 remaining (threshold=3.0774)
MiniCorrelationAnalysis - removed 3 expr (1569 remain), 0 mut (657 remain)
WeightedUnivar - 1519 expr removed, 50 expr remaining
SelectFeatures - mut selected: 50
  fold features: {'clin': 8, 'expr': 50, 'mut': 50}
fixed split: train 342 (198 labelled + 144 censored), test 50, TRACERx 150
  CEM  internal AUC 0.758 [0.615-0.878] | AUPRC 0.691 [0.496-0.900] | bal_acc 0.607 [0.473-0.740]
  CEM  external AUC 0.782 [0.708-0.850] | AUPRC 0.636 [0.496-0.765] | bal_acc 0.704 [0.628-0.780]
RemoveNearlyConstant - removed: 0 clin, 42 expr, 825 mut
RemoveRareMutations - 6530 mut removed (657 mut remaining, min_count=10)
RemoveLowExpression - 1476 expr removed (15718 expr remaining)
FilterVariance - 14146 expr removed, 1572 remaining (thre

In [25]:
plasma_fus = merge_results({'gated+att': plasma['CEM']}, {'gated-only': fus_abl['gated']['CEM'], 'concat': fus_abl['concat']['CEM']})
run_comparison(plasma_fus, 'fus_method', 'plasma_v_fus_method', ref='gated+att', secondary_metric='bss', order=['gated+att', 'gated-only', 'concat'])

TCGA test (n = 50) The no-information Brier score for this set, obtained by predicting its event rate for every patient, is 0.248.
TRACERx (n = 150) The no-information Brier score for this set, obtained by predicting its event rate for every patient, is 0.228.


,Cohort,Model,AUROC,AUPRC,Avg prec,Brier,BSS,Bal acc,Recall,Spec,Prec,Cal slope,Cal CIL
0,TCGA test,gated+att,0.816 (0.690-0.920),0.753 (0.545-0.922),0.766 (0.584-0.925),0.188 (0.153-0.225),0.25 (0.07-0.37),0.70 (0.57-0.82),0.74 (0.53-0.90),0.67 (0.48-0.84),0.65 (0.46-0.82),1.83 (0.93-3.51),-0.07 (-0.63 to 0.44)
1,TCGA test,gated-only,0.758 (0.615-0.878),0.691 (0.496-0.900),0.714 (0.534-0.903),0.198 (0.163-0.235),0.20 (0.02-0.33),0.61 (0.47-0.74),0.70 (0.48-0.88),0.52 (0.34-0.69),0.55 (0.36-0.72),1.58 (0.72-2.89),-0.04 (-0.63 to 0.48)
2,TCGA test,concat,0.713 (0.558-0.845),0.685 (0.466-0.872),0.698 (0.506-0.877),0.213 (0.178-0.245),0.14 (-0.02 to 0.26),0.66 (0.52-0.78),0.61 (0.39-0.80),0.70 (0.52-0.86),0.64 (0.44-0.83),1.62 (0.62-3.34),0.05 (-0.54 to 0.57)
3,TRACERx,gated+att,0.789 (0.714-0.857),0.661 (0.519-0.784),0.665 (0.536-0.788),0.185 (0.163-0.207),0.19 (0.07-0.28),0.70 (0.63-0.78),0.70 (0.57-0.81),0.71 (0.61-0.80),0.57 (0.45-0.70),1.56 (1.08-2.24),-0.31 (-0.67 to 0.03)
4,TRACERx,gated-only,0.782 (0.708-0.850),0.636 (0.496-0.765),0.642 (0.515-0.770),0.186 (0.163-0.209),0.19 (0.06-0.28),0.70 (0.63-0.78),0.72 (0.59-0.83),0.69 (0.60-0.78),0.56 (0.44-0.68),1.39 (0.95-2.01),-0.26 (-0.62 to 0.08)
5,TRACERx,concat,0.808 (0.734-0.874),0.666 (0.528-0.795),0.671 (0.540-0.800),0.188 (0.170-0.205),0.18 (0.06-0.26),0.75 (0.68-0.81),0.81 (0.70-0.91),0.69 (0.59-0.78),0.59 (0.48-0.70),2.08 (1.47-2.99),-0.38 (-0.72 to -0.05)


,Model,TCGA test Δ (95% CI),TCGA test perm. P,TCGA test perm. P (Holm),TCGA test DeLong P,TCGA test DeLong P (Holm),TRACERx Δ (95% CI),TRACERx perm. P,TRACERx perm. P (Holm),TRACERx DeLong P,TRACERx DeLong P (Holm),Pooled Δ (95% CI),Pooled P,Pooled P (Holm),Het. P
0,gated-only,0.058 (-0.006 to 0.140),0.068,0.068,0.111,0.111,0.007 (-0.018 to 0.033),0.592,0.751,0.564,0.636,0.013 (-0.011 to 0.037),0.302,0.604,0.200
1,concat,0.103 (0.016-0.203),0.017,0.034,0.027,0.054,-0.018 (-0.057 to 0.017),0.375,0.751,0.318,0.636,-0.002 (-0.037 to 0.032),0.891,0.891,0.019


,Model,TCGA test Δ (95% CI),TRACERx Δ (95% CI),Pooled Δ (95% CI),Pooled P,Pooled P (Holm),Het. P
0,gated-only,0.043 (-0.003 to 0.093),0.006 (-0.023 to 0.035),0.016 (-0.009 to 0.041),0.207,0.207,0.191
1,concat,0.101 (0.031-0.174),0.013 (-0.030 to 0.056),0.036 (-0.001 to 0.073),0.055,0.109,0.038


In [69]:
mod_abl = fixed_split_eval(lung_c, device, split_c, best, versions=('C', 'E', 'M', 'CE', 'CM', 'EM',), search_space=SPACES, sampler=lambda: optuna.samplers.GridSampler(GRID, seed=42), n_trials=n_trials, 
    k_inner=3, return_models=True, **fse_args)

save_run(mod_abl, 'mod_abl_oct20', kind='fixed_split', base=best, fold_kwargs=dict(scale_clin=True, expr_known_bonus=200, mut_boost_factor=3.5), split=split_c, lung=lung_c, known_genes=known_genes, grid=GRID)

RemoveNearlyConstant - removed: 0 clin, 42 expr, 825 mut
RemoveRareMutations - 6530 mut removed (657 mut remaining, min_count=10)
RemoveLowExpression - 1476 expr removed (15718 expr remaining)
FilterVariance - 14146 expr removed, 1572 remaining (threshold=3.0774)
MiniCorrelationAnalysis - removed 3 expr (1569 remain), 0 mut (657 remain)
WeightedUnivar - 1519 expr removed, 50 expr remaining
SelectFeatures - mut selected: 50
  fold features: {'clin': 8, 'expr': 50, 'mut': 50}
fixed split: train 342 (198 labelled + 144 censored), test 50, TRACERx 150
    inner best AUC 0.5628  params {'wd': 0.0003, 'enc_lr': 0.0006, 'fus_lr': 0.0003}
  C    tuned over 8 trials: inner-CV AUC 0.563  {'wd': 0.0003, 'enc_lr': 0.0006, 'fus_lr': 0.0003}
  C    internal AUC 0.678 [0.524-0.821] | AUPRC 0.700 [0.501-0.856] | bal_acc 0.643 [0.517-0.766]
  C    external AUC 0.768 [0.675-0.842] | AUPRC 0.633 [0.485-0.767] | bal_acc 0.718 [0.635-0.790]
  C    inner-CV 0.563 vs test 0.678 (gap -0.115: how optimistic th

'results/saved_runs/mod_abl_oct20.pkl'

In [29]:
mod_abl = load_run('mod_abl_oct20', lung=lung_c)
mod_abl = backfill_metrics(mod_abl['result'])
plasma_abl = merge_results({'CEM': plasma['CEM']}, mod_abl)

In [30]:
run_comparison(plasma_abl, 'mod_abl', 'plasma_v_mod_abl', ref='CEM', secondary_metric='bss', order=['CEM', 'CE', 'CM', 'EM', 'C', 'E', 'M'])

TCGA test (n = 50) The no-information Brier score for this set, obtained by predicting its event rate for every patient, is 0.248.
TRACERx (n = 150) The no-information Brier score for this set, obtained by predicting its event rate for every patient, is 0.228.


,Cohort,Model,AUROC,AUPRC,Avg prec,Brier,BSS,Bal acc,Recall,Spec,Prec,Cal slope,Cal CIL
0,TCGA test,CEM,0.816 (0.690-0.920),0.753 (0.545-0.922),0.766 (0.584-0.925),0.188 (0.153-0.225),0.25 (0.07-0.37),0.70 (0.57-0.82),0.74 (0.53-0.90),0.67 (0.48-0.84),0.65 (0.46-0.82),1.83 (0.93-3.51),-0.07 (-0.63 to 0.44)
1,TCGA test,CE,0.754 (0.601-0.880),0.760 (0.563-0.902),0.767 (0.587-0.908),0.202 (0.170-0.234),0.19 (0.02-0.30),0.70 (0.57-0.82),0.70 (0.50-0.87),0.70 (0.53-0.86),0.67 (0.48-0.85),1.95 (0.85-3.72),-0.01 (-0.56 to 0.51)
2,TCGA test,CM,0.726 (0.570-0.860),0.731 (0.536-0.880),0.738 (0.558-0.884),0.227 (0.208-0.245),0.09 (-0.02 to 0.13),0.63 (0.49-0.75),0.74 (0.55-0.91),0.52 (0.32-0.71),0.57 (0.38-0.74),4.72 (2.21-9.07),0.03 (-0.53 to 0.55)
3,TCGA test,EM,0.659 (0.505-0.802),0.637 (0.416-0.814),0.648 (0.455-0.826),0.232 (0.194-0.271),0.06 (-0.14 to 0.21),0.59 (0.46-0.72),0.52 (0.30-0.74),0.67 (0.48-0.83),0.57 (0.35-0.78),0.83 (0.00-1.81),-0.09 (-0.71 to 0.49)
4,TCGA test,C,0.678 (0.524-0.821),0.700 (0.501-0.856),0.707 (0.531-0.860),0.231 (0.214-0.249),0.07 (-0.03 to 0.12),0.64 (0.52-0.77),0.43 (0.24-0.64),0.85 (0.69-0.97),0.71 (0.45-0.93),3.73 (1.30-7.20),0.02 (-0.55 to 0.55)
5,TCGA test,E,0.678 (0.532-0.819),0.676 (0.464-0.839),0.684 (0.492-0.845),0.225 (0.183-0.264),0.09 (-0.12 to 0.25),0.61 (0.48-0.73),0.74 (0.53-0.91),0.48 (0.29-0.66),0.55 (0.38-0.71),0.92 (0.21-1.87),-0.11 (-0.72 to 0.46)
6,TCGA test,M,0.535 (0.365-0.694),0.525 (0.327-0.723),0.537 (0.360-0.736),0.252 (0.225-0.279),-0.01 (-0.17 to 0.08),0.53 (0.39-0.66),0.39 (0.20-0.60),0.67 (0.48-0.83),0.50 (0.26-0.73),0.27 (-1.32 to 1.97),-0.01 (-0.63 to 0.54)
7,TRACERx,CEM,0.789 (0.714-0.857),0.661 (0.519-0.784),0.665 (0.536-0.788),0.185 (0.163-0.207),0.19 (0.07-0.28),0.70 (0.63-0.78),0.70 (0.57-0.81),0.71 (0.61-0.80),0.57 (0.45-0.70),1.56 (1.08-2.24),-0.31 (-0.67 to 0.03)
8,TRACERx,CE,0.777 (0.697-0.850),0.653 (0.504-0.774),0.658 (0.523-0.781),0.190 (0.172-0.209),0.17 (0.06-0.25),0.71 (0.63-0.78),0.64 (0.51-0.76),0.77 (0.68-0.85),0.61 (0.48-0.74),1.85 (1.28-2.68),-0.32 (-0.67 to 0.00)
9,TRACERx,CM,0.757 (0.667-0.835),0.608 (0.455-0.755),0.616 (0.475-0.761),0.219 (0.209-0.229),0.04 (-0.09 to 0.11),0.63 (0.57-0.70),0.87 (0.77-0.96),0.40 (0.30-0.50),0.44 (0.35-0.54),3.60 (2.22-5.49),-0.50 (-0.84 to -0.19)


,Model,TCGA test Δ (95% CI),TCGA test perm. P,TCGA test perm. P (Holm),TCGA test DeLong P,TCGA test DeLong P (Holm),TRACERx Δ (95% CI),TRACERx perm. P,TRACERx perm. P (Holm),TRACERx DeLong P,TRACERx DeLong P (Holm),Pooled Δ (95% CI),Pooled P,Pooled P (Holm),Het. P
0,CE,0.063 (0.000-0.148),0.022,0.065,0.082,0.246,0.012 (-0.011 to 0.036),0.327,0.980,0.295,0.885,0.017 (-0.005 to 0.040),0.138,0.413,0.197
1,CM,0.090 (-0.043 to 0.234),0.231,0.231,0.196,0.246,0.032 (-0.038 to 0.105),0.475,0.980,0.374,0.885,0.045 (-0.019 to 0.109),0.170,0.413,0.469
2,EM,0.158 (0.054-0.276),0.004,0.022,0.005,0.024,0.076 (0.011-0.141),0.024,0.096,0.022,0.086,0.097 (0.040-0.153),< 0.001,0.003,0.213
3,C,0.138 (-0.021 to 0.304),0.108,0.217,0.096,0.246,0.022 (-0.052 to 0.098),0.655,0.980,0.570,0.885,0.042 (-0.026 to 0.110),0.227,0.413,0.202
4,E,0.138 (0.037-0.256),0.010,0.040,0.011,0.044,0.101 (0.029-0.177),0.007,0.037,0.006,0.029,0.113 (0.052-0.173),< 0.001,0.001,0.579
5,M,0.281 (0.102-0.453),0.002,0.013,0.002,0.010,0.316 (0.194-0.436),< 0.001,< 0.001,< 0.001,< 0.001,0.304 (0.205-0.404),< 0.001,< 0.001,0.748


,Model,TCGA test Δ (95% CI),TRACERx Δ (95% CI),Pooled Δ (95% CI),Pooled P,Pooled P (Holm),Het. P
0,CE,0.058 (0.010-0.102),0.023 (-0.002 to 0.049),0.031 (0.009-0.053),0.007,0.007,0.206
1,CM,0.159 (0.024-0.290),0.150 (0.068-0.237),0.152 (0.082-0.223),< 0.001,< 0.001,0.913
2,EM,0.181 (0.088-0.297),0.093 (0.024-0.160),0.119 (0.062-0.176),< 0.001,< 0.001,0.163
3,C,0.175 (0.037-0.311),0.149 (0.064-0.240),0.156 (0.083-0.229),< 0.001,< 0.001,0.749
4,E,0.150 (0.046-0.276),0.094 (0.018-0.172),0.112 (0.047-0.176),< 0.001,0.001,0.424
5,M,0.260 (0.107-0.411),0.346 (0.225-0.485),0.310 (0.212-0.408),< 0.001,< 0.001,0.395
